In [ ]:
RETRAIN = False


# PatchCore Sweep Notebook (Pretrained ResNet18 Backbone)

This notebook is the curated submission entry point for the `x64` ResNet18 PatchCore sweep.

Default behavior:
- reuse the saved sweep summary, selected checkpoint, and saved evaluation CSVs when they already exist
- rerun the full PatchCore sweep only if `RETRAIN = True`
- display the main figures in the notebook and save them into the experiment-local artifact folder

## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/patchcore/resnet18/x64/main/train_config.toml`
- Artifact root: `experiments/anomaly_detection/patchcore/resnet18/x64/main/artifacts/patchcore_resnet18`
- Canonical checkpoint path: `experiments/anomaly_detection/patchcore/resnet18/x64/main/artifacts/patchcore_resnet18/checkpoints/best_model.pt`
- Default mode: artifact-first evaluation and visualization

## Imports and Repo Discovery

This cell resolves the repository root, imports the shared PatchCore utilities, and prepares the plotting/data libraries used throughout the notebook.

In [1]:
from pathlib import Path
import copy
import json
import random
import shutil
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.patchcore import PatchCoreModel
from wafer_defect.training.patchcore import build_memory_subset, collect_memory_bank

## Configuration and Artifact Policy

This cell loads the local experiment config and defines the small PatchCore sweep. `RETRAIN` stays `False` by default so a grader can run the notebook without retraining.

In [2]:
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/patchcore/resnet18/x64/main/train_config.toml'
config = load_toml(CONFIG_PATH)
PATCHCORE_SWEEP = [{'name': 'mean_mb10k', 'memory_bank_size': 10000, 'reduction': 'mean', 'topk_ratio': 0.1}, {'name': 'mean_mb50k', 'memory_bank_size': 50000, 'reduction': 'mean', 'topk_ratio': 0.1}, {'name': 'topk_mb10k_r005', 'memory_bank_size': 10000, 'reduction': 'topk_mean', 'topk_ratio': 0.05}, {'name': 'topk_mb50k_r005', 'memory_bank_size': 50000, 'reduction': 'topk_mean', 'topk_ratio': 0.05}, {'name': 'topk_mb50k_r010', 'memory_bank_size': 50000, 'reduction': 'topk_mean', 'topk_ratio': 0.1}, {'name': 'max_mb50k', 'memory_bank_size': 50000, 'reduction': 'max', 'topk_ratio': 0.1}]
SELECTED_VARIANT_NAME = None
AUTO_SELECT_METRIC = 'f1'
RENDER_ALL_SAVED_VARIANTS = True
VARIANTS_TO_RENDER: list[str] = []
artifact_root = REPO_ROOT / config['run']['output_dir']
checkpoints_dir = artifact_root / 'checkpoints'
results_dir = artifact_root / 'results'
evaluation_dir = results_dir / 'evaluation'
plots_dir = artifact_root / 'plots'
VARIANT_COLOR_VAL = '#4d908e'
VARIANT_COLOR_NORMAL = '#577590'
VARIANT_COLOR_ANOMALY = '#f3722c'
VARIANT_COLOR_DEFECT = '#43aa8b'
for directory in [artifact_root, checkpoints_dir, results_dir, evaluation_dir, plots_dir]:
    directory.mkdir(parents=True, exist_ok=True)
config


{'run': {'output_dir': 'experiments/anomaly_detection/patchcore/resnet18/x64/main/artifacts/patchcore_resnet18',
  'seed': 42},
 'data': {'metadata_csv': 'data/processed/x64/wm811k/metadata_50k_5pct.csv',
  'image_size': 64,
  'batch_size': 64,
  'num_workers': 0},
 'training': {'device': 'auto'},
 'model': {'type': 'patchcore',
  'backbone_type': 'resnet18',
  'pretrained': True,
  'freeze_backbone': True,
  'backbone_input_size': 224,
  'normalize_imagenet': True,
  'memory_bank_size': 50000,
  'reduction': 'mean',
  'topk_ratio': 0.1,
  'query_chunk_size': 2048,
  'memory_chunk_size': 8192}}

## Runtime Setup

This cell fixes random seeds and resolves the compute device so reruns stay consistent.

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)
seed = int(config['run']['seed'])
set_seed(seed)
device = resolve_device(config['training'].get('device', 'auto'))
device

device(type='cuda')

## Dataset Loading

This cell loads the shared processed `64x64` benchmark split and prepares dataloaders for validation and test scoring.

In [4]:
image_size = int(config['data'].get('image_size', 64))
batch_size = int(config['data'].get('batch_size', 64))
num_workers = int(config['data'].get('num_workers', 0))
metadata_path = REPO_ROOT / config['data']['metadata_csv']
metadata = pd.read_csv(metadata_path)
display(metadata.head())
display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
display(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))
train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')


,array_path,label,defect_type,is_anomaly,split,source_split,original_height,original_width
0,data/processed/x64/wm811k/arrays_50k_5pct/wafe...,none,none,0,train,Training,25,27
1,data/processed/x64/wm811k/arrays_50k_5pct/wafe...,none,none,0,train,Training,55,66
2,data/processed/x64/wm811k/arrays_50k_5pct/wafe...,none,none,0,train,Test,33,29
3,data/processed/x64/wm811k/arrays_50k_5pct/wafe...,none,none,0,train,Training,25,26
4,data/processed/x64/wm811k/arrays_50k_5pct/wafe...,none,none,0,train,Test,39,37


,count
split,
train,40000
test,5250
val,5000


,count
is_anomaly,
0,50000
1,250


train=40000, val=5000, test=5250


## Model and Sweep Helpers

These helpers build the PatchCore model, reuse cached memory banks during reruns, and load previously saved variant outputs when retraining is not requested.

In [ ]:
base_model_config = config['model']
memory_bank_cache: dict[int, dict[str, object]] = {}

def build_patchcore_model(*, reduction: str, topk_ratio: float) -> PatchCoreModel:
    return PatchCoreModel(image_size=image_size, backbone_type=str(base_model_config.get('backbone_type', 'resnet18')), use_batchnorm=bool(base_model_config.get('use_batchnorm', True)), pretrained=bool(base_model_config.get('pretrained', True)), freeze_backbone=bool(base_model_config.get('freeze_backbone', True)), backbone_input_size=int(base_model_config.get('backbone_input_size', 224)), normalize_imagenet=bool(base_model_config.get('normalize_imagenet', True)), reduction=str(reduction), topk_ratio=float(topk_ratio), query_chunk_size=int(base_model_config.get('query_chunk_size', 2048)), memory_chunk_size=int(base_model_config.get('memory_chunk_size', 8192))).to(device)

def get_memory_bank_info(memory_bank_size: int) -> dict[str, object]:
    if memory_bank_size not in memory_bank_cache:
        temp_model = build_patchcore_model(reduction='mean', topk_ratio=0.1)
        memory_subset = build_memory_subset(train_dataset, memory_bank_size=memory_bank_size, patches_per_image=temp_model.patches_per_image, seed=seed)
        memory_loader = DataLoader(memory_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        memory_bank = collect_memory_bank(model=temp_model, dataloader=memory_loader, device=device, target_size=memory_bank_size, seed=seed)
        memory_bank_cache[memory_bank_size] = {'memory_bank': memory_bank.cpu(), 'memory_subset_images': len(memory_subset), 'patches_per_image': int(temp_model.patches_per_image), 'feature_dim': int(temp_model.feature_dim)}
        del temp_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return memory_bank_cache[memory_bank_size]

def collect_scores(model: PatchCoreModel, dataloader: DataLoader) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            scores = model(inputs).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                rows.append({'score': float(score), 'is_anomaly': int(label)})
    return pd.DataFrame(rows)

def first_existing_path(*candidates: Path) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def load_json_file(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def load_variant_outputs(variant_name: str) -> dict[str, object]:
    run_output_dir = artifact_root / variant_name
    summary_path = first_existing_path(run_output_dir / 'results' / 'summary.json', run_output_dir / 'summary.json')
    val_scores_path = first_existing_path(run_output_dir / 'results' / 'evaluation' / 'val_scores.csv', run_output_dir / 'evaluation' / 'val_scores.csv')
    test_scores_path = first_existing_path(run_output_dir / 'results' / 'evaluation' / 'test_scores.csv', run_output_dir / 'evaluation' / 'test_scores.csv')
    threshold_sweep_path = first_existing_path(run_output_dir / 'results' / 'evaluation' / 'threshold_sweep.csv', run_output_dir / 'evaluation' / 'threshold_sweep.csv')
    if not all([summary_path, val_scores_path, test_scores_path, threshold_sweep_path]):
        missing = {'summary_path': summary_path, 'val_scores_path': val_scores_path, 'test_scores_path': test_scores_path, 'threshold_sweep_path': threshold_sweep_path}
        print(f'[WARNING] Missing cached files for variant {variant_name}: {missing}')
        return None
    summary = load_json_file(summary_path)
    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)
    threshold = float(summary['threshold'])
    labels = test_scores_df['is_anomaly'].to_numpy()
    scores = test_scores_df['score'].to_numpy()
    metrics = summarize_threshold_metrics(labels, scores, threshold)
    best_sweep = threshold_sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
    return {'summary': summary, 'val_scores_df': val_scores_df, 'test_scores_df': test_scores_df, 'threshold_sweep_df': threshold_sweep_df, 'metrics': metrics, 'best_sweep': best_sweep, 'output_dir': run_output_dir}

def compute_failure_tables(test_metadata: pd.DataFrame, test_scores_df: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    analysis_df = test_metadata.reset_index(drop=True).copy()
    analysis_df['score'] = test_scores_df.reset_index(drop=True)['score']
    analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)
    analysis_df['error_type'] = 'tn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
    error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
    defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
    defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
    return (analysis_df, error_summary_df, defect_recall_df)

def render_variant_artifacts(variant_name: str, variant_payload: dict[str, object]) -> dict[str, str]:
    summary = variant_payload['summary']
    val_scores_df = variant_payload['val_scores_df']
    test_scores_df = variant_payload['test_scores_df']
    threshold_sweep_df = variant_payload['threshold_sweep_df']
    metrics = variant_payload['metrics']
    best_sweep = variant_payload['best_sweep']
    threshold = float(summary['threshold'])
    variant_root = variant_payload['output_dir']
    variant_plots_dir = variant_root / 'plots'
    variant_results_dir = variant_root / 'results'
    variant_evaluation_dir = variant_results_dir / 'evaluation'
    variant_plots_dir.mkdir(parents=True, exist_ok=True)
    variant_evaluation_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color=VARIANT_COLOR_VAL)
    axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[0].set_title(f'Validation Normal Score Distribution\n{variant_name}')
    axes[0].legend()
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color=VARIANT_COLOR_NORMAL)
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color=VARIANT_COLOR_ANOMALY)
    axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[1].set_title(f'Test Score Distribution\n{variant_name}')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
    ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
    ax.set_title(f'Threshold Sweep on Test Split\n{variant_name}')
    ax.set_xlabel('Anomaly-score threshold')
    ax.set_ylabel('Metric value')
    ax.legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    cm_array = np.asarray(metrics['confusion_matrix'], dtype=float)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_array, cmap='Blues')
    ax.set_xticks([0, 1], labels=['pred_normal', 'pred_anomaly'])
    ax.set_yticks([0, 1], labels=['true_normal', 'true_anomaly'])
    ax.set_title(f'Confusion Matrix\n{variant_name}')
    for row_idx in range(cm_array.shape[0]):
        for col_idx in range(cm_array.shape[1]):
            ax.text(col_idx, row_idx, int(cm_array[row_idx, col_idx]), ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_dataset.metadata, test_scores_df, threshold)
    analysis_df.to_csv(variant_evaluation_dir / 'analysis_with_predictions.csv', index=False)
    error_summary_df.reset_index().to_csv(variant_evaluation_dir / 'error_summary.csv', index=False)
    defect_recall_df.reset_index().to_csv(variant_evaluation_dir / 'defect_recall.csv', index=False)
    top_defects_df = defect_recall_df.head(10).reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color=VARIANT_COLOR_ANOMALY)
    axes[0].set_title(f'Prediction Outcome Counts\n{variant_name}')
    axes[0].set_ylabel('count')
    axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color=VARIANT_COLOR_DEFECT)
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title('Top Defect-Type Recall')
    axes[1].set_xlabel('recall')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    return {'plots_dir': str(variant_plots_dir), 'evaluation_dir': str(variant_evaluation_dir)}

def resolve_variant_names_to_render(sweep_results_df: pd.DataFrame, selected_variant_name: str) -> list[str]:
    names = []
    if RENDER_ALL_SAVED_VARIANTS:
        names.extend(sweep_results_df['name'].astype(str).tolist())
    names.extend([str(name) for name in VARIANTS_TO_RENDER])
    names.append(str(selected_variant_name))
    ordered = []
    seen = set()
    for name in names:
        if name not in seen:
            ordered.append(name)
            seen.add(name)
    return ordered
print({'sweep_variants': [variant['name'] for variant in PATCHCORE_SWEEP], 'artifact_root': str(artifact_root), 'backbone_type': str(base_model_config.get('backbone_type', 'resnet18')), 'default_mode': 'artifact reuse' if not RETRAIN else 'full sweep rerun'})


## Load Cached Sweep Results or Rerun the Sweep

This cell is the heart of the notebook. It loads the saved sweep and selected variant by default, but it can rerun the whole PatchCore sweep and refresh the artifacts if you explicitly

In [6]:
selected_variant_name = None
selected_variant = None
selected_variant_name = None
selected_variant = None
selected_variant_name = None
selected_variant = None
sweep_results_path = first_existing_path(results_dir / 'patchcore_sweep_results.csv', artifact_root / 'patchcore_sweep_results.csv')
sweep_summary_path = first_existing_path(results_dir / 'patchcore_sweep_summary.json', artifact_root / 'patchcore_sweep_summary.json')
use_cached_outputs = not RETRAIN and sweep_results_path is not None and (sweep_summary_path is not None)
variant_outputs = {}
ranking_metrics = [AUTO_SELECT_METRIC, *[metric for metric in ['f1', 'auroc', 'auprc'] if metric != AUTO_SELECT_METRIC]]
if use_cached_outputs:
    sweep_results_df = pd.read_csv(sweep_results_path).sort_values(ranking_metrics, ascending=False).reset_index(drop=True)
    sweep_summary = load_json_file(sweep_summary_path)
    selected_variant_name = str(SELECTED_VARIANT_NAME or sweep_summary.get('selected_variant_name') or sweep_results_df.iloc[0]['name'])
    selected_variant = load_variant_outputs(selected_variant_name)
    if selected_variant is None:
        print('[WARNING] Cached sweep outputs exist, but the selected variant bundle is incomplete.')
        sweep_results_df = pd.DataFrame()
        sweep_summary = {}
        selected_variant_name = None
    else:
        variant_outputs[selected_variant_name] = selected_variant
        print(f'Loaded cached PatchCore sweep results from {sweep_results_path}')
elif RETRAIN:
    sweep_rows = []
    for variant in PATCHCORE_SWEEP:
        variant_name = str(variant['name'])
        run_output_dir = artifact_root / variant_name
        variant_checkpoints_dir = run_output_dir / 'checkpoints'
        variant_results_dir = run_output_dir / 'results'
        variant_evaluation_dir = variant_results_dir / 'evaluation'
        for directory in [run_output_dir, variant_checkpoints_dir, variant_results_dir, variant_evaluation_dir]:
            directory.mkdir(parents=True, exist_ok=True)
        print(f'\n=== PatchCore variant: {variant_name} ===')
        model = build_patchcore_model(reduction=variant['reduction'], topk_ratio=variant['topk_ratio'])
        memory_info = get_memory_bank_info(int(variant['memory_bank_size']))
        model.set_memory_bank(memory_info['memory_bank'].to(device))
        run_config = copy.deepcopy(config)
        run_config['run']['output_dir'] = run_output_dir.relative_to(REPO_ROOT).as_posix()
        run_config['model']['memory_bank_size'] = int(variant['memory_bank_size'])
        run_config['model']['reduction'] = str(variant['reduction'])
        run_config['model']['topk_ratio'] = float(variant['topk_ratio'])
        checkpoint = {'model_state_dict': model.state_dict(), 'config': run_config, 'memory_bank_size': int(model.memory_bank.shape[0]), 'feature_dim': int(model.feature_dim), 'patches_per_image': int(model.patches_per_image), 'backbone_type': str(base_model_config.get('backbone_type', 'resnet18'))}
        torch.save(checkpoint, variant_checkpoints_dir / 'best_model.pt')
        torch.save(checkpoint, variant_checkpoints_dir / 'last_model.pt')
        val_scores_df = collect_scores(model, val_loader)
        test_scores_df = collect_scores(model, test_loader)
        val_normal_scores = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score']
        threshold = float(val_normal_scores.quantile(0.95))
        labels = test_scores_df['is_anomaly'].to_numpy()
        scores = test_scores_df['score'].to_numpy()
        metrics = summarize_threshold_metrics(labels, scores, threshold)
        threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)
        summary = {'name': variant_name, 'memory_bank_size': int(model.memory_bank.shape[0]), 'memory_subset_images': int(memory_info['memory_subset_images']), 'patches_per_image': int(memory_info['patches_per_image']), 'feature_dim': int(memory_info['feature_dim']), 'reduction': str(variant['reduction']), 'topk_ratio': float(variant['topk_ratio']), 'threshold': threshold, 'precision': float(metrics['precision']), 'recall': float(metrics['recall']), 'f1': float(metrics['f1']), 'auroc': float(metrics['auroc']), 'auprc': float(metrics['auprc']), 'best_sweep_threshold': float(best_sweep['threshold']), 'best_sweep_precision': float(best_sweep['precision']), 'best_sweep_recall': float(best_sweep['recall']), 'best_sweep_f1': float(best_sweep['f1']), 'predicted_anomalies': int(metrics['predicted_anomalies']), 'output_dir': run_output_dir.relative_to(REPO_ROOT).as_posix()}
        val_scores_df.to_csv(variant_evaluation_dir / 'val_scores.csv', index=False)
        test_scores_df.to_csv(variant_evaluation_dir / 'test_scores.csv', index=False)
        threshold_sweep_df.to_csv(variant_evaluation_dir / 'threshold_sweep.csv', index=False)
        (variant_results_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
        sweep_rows.append(summary)
        variant_outputs[variant_name] = {'summary': summary, 'val_scores_df': val_scores_df, 'test_scores_df': test_scores_df, 'threshold_sweep_df': threshold_sweep_df, 'metrics': metrics, 'best_sweep': best_sweep, 'output_dir': run_output_dir}
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    sweep_results_df = pd.DataFrame(sweep_rows).sort_values(ranking_metrics, ascending=False).reset_index(drop=True)
    sweep_results_df.to_csv(results_dir / 'patchcore_sweep_results.csv', index=False)
    selected_variant_name = str(SELECTED_VARIANT_NAME or sweep_results_df.iloc[0]['name'])
    selected_row = sweep_results_df.loc[sweep_results_df['name'] == selected_variant_name].iloc[0].to_dict()
    sweep_summary = {'selected_variant_name': selected_variant_name, 'auto_select_metric': AUTO_SELECT_METRIC, 'ranking_metrics': ranking_metrics, 'selected_row': selected_row, 'results': sweep_rows}
    (results_dir / 'patchcore_sweep_summary.json').write_text(json.dumps(sweep_summary, indent=2), encoding='utf-8')
    selected_variant = variant_outputs[selected_variant_name]
    shutil.copy2(selected_variant['output_dir'] / 'checkpoints' / 'best_model.pt', checkpoints_dir / 'best_model.pt')
    shutil.copy2(selected_variant['output_dir'] / 'checkpoints' / 'last_model.pt', checkpoints_dir / 'last_model.pt')
    shutil.copy2(selected_variant['output_dir'] / 'results' / 'summary.json', results_dir / 'summary.json')
    selected_variant['val_scores_df'].to_csv(evaluation_dir / 'val_scores.csv', index=False)
    selected_variant['test_scores_df'].to_csv(evaluation_dir / 'test_scores.csv', index=False)
    selected_variant['threshold_sweep_df'].to_csv(evaluation_dir / 'threshold_sweep.csv', index=False)
    (evaluation_dir / 'summary.json').write_text(json.dumps(selected_variant['summary'], indent=2), encoding='utf-8')
    print(f'Finished rerunning PatchCore sweep. Selected variant: {selected_variant_name}')
else:
    sweep_results_df = pd.DataFrame()
    sweep_summary = {}
    selected_variant_name = None
    selected_variant = None
    print('[WARNING] RETRAIN is False and the saved PatchCore sweep outputs are missing. Skipping the sweep rerun.')
display(sweep_results_df)


[WARNING] RETRAIN is False and the saved PatchCore sweep outputs are missing. Skipping the sweep rerun.


""


## Selected Variant Metrics

This cell loads the selected variant into a compact metrics table and confusion matrix so the notebook immediately shows the main benchmark numbers.

In [7]:
if selected_variant_name is not None and selected_variant is not None:
    if selected_variant_name is not None and selected_variant is not None:
        if selected_variant_name is not None and selected_variant is not None:
            selected_variant = variant_outputs[selected_variant_name]
            output_dir = selected_variant['output_dir']
            summary = selected_variant['summary']
            val_scores_df = selected_variant['val_scores_df']
            test_scores_df = selected_variant['test_scores_df']
            threshold_sweep_df = selected_variant['threshold_sweep_df']
            metrics = selected_variant['metrics']
            best_sweep = selected_variant['best_sweep']
            threshold = float(summary['threshold'])
            confusion_df = pd.DataFrame(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
            metrics_df = pd.DataFrame([{'metric': 'precision', 'value': metrics['precision']}, {'metric': 'recall', 'value': metrics['recall']}, {'metric': 'f1', 'value': metrics['f1']}, {'metric': 'auroc', 'value': metrics['auroc']}, {'metric': 'auprc', 'value': metrics['auprc']}, {'metric': 'threshold', 'value': threshold}])
            display(metrics_df)
            display(confusion_df)
            print(f'Selected variant: {selected_variant_name}')
            print(f"Canonical checkpoint: {checkpoints_dir / 'best_model.pt'}")
            print(f"Best sweep threshold: {best_sweep['threshold']:.6f} | precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}")
        else:
            print('Selected-variant metrics are unavailable because the cached sweep artifacts are missing.')
    else:
        print('Selected-variant metrics are unavailable because the cached sweep artifacts are missing.')
else:
    print('Selected-variant metrics are unavailable because the cached sweep artifacts are missing.')


Selected-variant metrics are unavailable because the cached sweep artifacts are missing.


## Visualizations

These plots summarize the sweep ranking, score distributions, threshold sensitivity, and confusion matrix. Each figure is displayed inline and saved under `plots/`.

In [8]:
if selected_variant_name is not None and selected_variant is not None:
    if selected_variant_name is not None and selected_variant is not None:
        if selected_variant_name is not None and selected_variant is not None:
            plot_df = sweep_results_df.copy()
            plot_df['label'] = plot_df['name'] + '\n' + plot_df['reduction'] + ', mb=' + plot_df['memory_bank_size'].astype(str)
            fig, axes = plt.subplots(1, 2, figsize=(16, 5))
            axes[0].barh(plot_df['label'], plot_df['f1'], color='#277da1')
            axes[0].set_title('PatchCore ResNet18 Sweep: Validation-Threshold F1')
            axes[0].set_xlabel('F1')
            axes[0].invert_yaxis()
            axes[1].barh(plot_df['label'], plot_df['auroc'], color='#90be6d')
            axes[1].set_title('PatchCore ResNet18 Sweep: AUROC')
            axes[1].set_xlabel('AUROC')
            axes[1].invert_yaxis()
            plt.tight_layout()
            fig.savefig(plots_dir / 'variant_comparison_metrics.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color='#4d908e')
            axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
            axes[0].set_title(f'Validation Normal Score Distribution\n{selected_variant_name}')
            axes[0].legend()
            axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color='#577590')
            axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color='#f3722c')
            axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
            axes[1].set_title(f'Test Score Distribution\n{selected_variant_name}')
            axes[1].legend()
            plt.tight_layout()
            fig.savefig(plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
            ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
            ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
            ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
            ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
            ax.set_title(f'Threshold Sweep on Test Split\n{selected_variant_name}')
            ax.set_xlabel('Anomaly-score threshold')
            ax.set_ylabel('Metric value')
            ax.legend()
            plt.tight_layout()
            fig.savefig(plots_dir / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            cm_array = np.asarray(metrics['confusion_matrix'], dtype=float)
            fig, ax = plt.subplots(figsize=(5, 4))
            im = ax.imshow(cm_array, cmap='Blues')
            ax.set_xticks([0, 1], labels=['pred_normal', 'pred_anomaly'])
            ax.set_yticks([0, 1], labels=['true_normal', 'true_anomaly'])
            ax.set_title(f'Confusion Matrix\n{selected_variant_name}')
            for row_idx in range(cm_array.shape[0]):
                for col_idx in range(cm_array.shape[1]):
                    ax.text(col_idx, row_idx, int(cm_array[row_idx, col_idx]), ha='center', va='center', color='black')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.tight_layout()
            fig.savefig(plots_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
        else:
            print('Selected-variant plots are unavailable because the cached sweep artifacts are missing.')
    else:
        print('Selected-variant plots are unavailable because the cached sweep artifacts are missing.')
else:
    print('Selected-variant plots are unavailable because the cached sweep artifacts are missing.')


Selected-variant plots are unavailable because the cached sweep artifacts are missing.


## Failure Analysis Tables

This cell attaches the selected PatchCore scores to the test metadata, saves the analysis tables into `results/evaluation/`, and surfaces the main false-positive and false-negative patterns.

In [9]:
if selected_variant_name is not None and selected_variant is not None:
    if selected_variant_name is not None and selected_variant is not None:
        if selected_variant_name is not None and selected_variant is not None:
            analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
            analysis_df['score'] = test_scores_df.reset_index(drop=True)['score']
            analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)
            analysis_df['error_type'] = 'tn'
            analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
            analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
            analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
            error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
            defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
            defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
            analysis_df.to_csv(evaluation_dir / 'analysis_with_predictions.csv', index=False)
            error_summary_df.reset_index().to_csv(evaluation_dir / 'error_summary.csv', index=False)
            defect_recall_df.reset_index().to_csv(evaluation_dir / 'defect_recall.csv', index=False)
            print(f'Failure analysis variant: {selected_variant_name}')
            display(error_summary_df)
            display(defect_recall_df)
            analysis_df.head()
        else:
            print('Failure-analysis tables are unavailable because the cached sweep artifacts are missing.')
    else:
        print('Failure-analysis tables are unavailable because the cached sweep artifacts are missing.')
else:
    print('Failure-analysis tables are unavailable because the cached sweep artifacts are missing.')


Failure-analysis tables are unavailable because the cached sweep artifacts are missing.


## Failure Analysis Plot

This final figure turns the saved failure tables into a quick visual summary for the report and for notebook readers.

In [10]:
if selected_variant_name is not None and selected_variant is not None:
    if selected_variant_name is not None and selected_variant is not None:
        if selected_variant_name is not None and selected_variant is not None:
            top_defects_df = defect_recall_df.head(10).reset_index()
            fig, axes = plt.subplots(1, 2, figsize=(15, 5))
            axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color='#f3722c')
            axes[0].set_title(f'Prediction Outcome Counts\n{selected_variant_name}')
            axes[0].set_ylabel('count')
            axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color='#43aa8b')
            axes[1].set_xlim(0.0, 1.0)
            axes[1].set_title('Top Defect-Type Recall')
            axes[1].set_xlabel('recall')
            axes[1].invert_yaxis()
            plt.tight_layout()
            fig.savefig(plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
        else:
            print('Failure-analysis plots are unavailable because the cached sweep artifacts are missing.')
    else:
        print('Failure-analysis plots are unavailable because the cached sweep artifacts are missing.')
else:
    print('Failure-analysis plots are unavailable because the cached sweep artifacts are missing.')


Failure-analysis plots are unavailable because the cached sweep artifacts are missing.


## Cached Variant Rendering

This section can populate each saved variant folder with its own plots and failure-analysis CSVs using the cached score files. By default it renders all saved sweep variants without retraining.

In [11]:
if selected_variant_name is None or selected_variant is None or sweep_results_df.empty:
    rendered_variants_df = pd.DataFrame(columns=['variant_name', 'plots_dir', 'evaluation_dir'])
    print('[WARNING] Cached variant rendering is being skipped because the sweep outputs are incomplete.')
else:
    variant_names_to_render = resolve_variant_names_to_render(sweep_results_df, selected_variant_name)
    rendered_variant_rows = []
    for variant_name in variant_names_to_render:
        if variant_name not in variant_outputs:
            variant_outputs[variant_name] = load_variant_outputs(variant_name)
        variant_payload = variant_outputs.get(variant_name)
        if variant_payload is None:
            print(f'[WARNING] Skipping variant rendering for {variant_name} because its cached files are incomplete.')
            continue
        render_info = render_variant_artifacts(variant_name, variant_payload)
        rendered_variant_rows.append({'variant_name': variant_name, 'plots_dir': render_info['plots_dir'], 'evaluation_dir': render_info['evaluation_dir']})
    rendered_variants_df = pd.DataFrame(rendered_variant_rows)
    display(rendered_variants_df)


[WARNING] Cached variant rendering is being skipped because the sweep outputs are incomplete.


## Saved Outputs

This cell prints the final artifact locations so the notebook doubles as reproducibility documentation.

In [12]:
if selected_variant_name is not None and selected_variant is not None:
    if selected_variant_name is not None and selected_variant is not None:
        if selected_variant_name is not None and selected_variant is not None:
            saved_outputs = {'checkpoint_dir': str(checkpoints_dir), 'results_dir': str(results_dir), 'evaluation_dir': str(evaluation_dir), 'plots_dir': str(plots_dir), 'selected_variant_name': selected_variant_name, 'rendered_variants': rendered_variants_df['variant_name'].tolist()}
            saved_outputs
        else:
            print('Saved-output summary is unavailable because the selected variant did not load.')
    else:
        print('Saved-output summary is unavailable because the selected variant did not load.')
else:
    print('Saved-output summary is unavailable because the selected variant did not load.')


Saved-output summary is unavailable because the selected variant did not load.


## UMAP Review

This merged section keeps the saved UMAP evaluation alongside the main PatchCore sweep notebook. It loads the selected variant's exported UMAP points when they exist and otherwise logs a warning instead of stopping the notebook.


In [13]:
selected_umap_df = None
selected_umap_path = None
if selected_variant_name is None or selected_variant is None:
    print('[WARNING] The selected variant did not load, so the saved UMAP review is unavailable.')
else:
    selected_eval_dir = selected_variant['output_dir'] / 'results' / 'evaluation'
    umap_candidates = [selected_eval_dir / 'plots' / 'embedding_umap_points.csv', selected_eval_dir / 'embedding_umap_points.csv']
    selected_umap_path = next((path for path in umap_candidates if path.exists()), None)
    if selected_umap_path is None:
        print('[WARNING] No saved ResNet18 PatchCore UMAP CSV was found for the selected variant.')
        for candidate in umap_candidates:
            print('  -', candidate)
    else:
        selected_umap_df = pd.read_csv(selected_umap_path)
        print(f'Loaded selected-variant UMAP points from {selected_umap_path}')
        display(selected_umap_df.head())
        display(selected_umap_df['split_label'].value_counts(dropna=False).rename('count').to_frame())


[WARNING] The selected variant did not load, so the saved UMAP review is unavailable.


In [14]:
embedded_anomalies = pd.DataFrame()
selected_umap_test_df = None
if selected_umap_df is None:
    print('[WARNING] UMAP neighborhood analysis is unavailable because no saved UMAP CSV was found.')
else:
    from sklearn.neighbors import NearestNeighbors
    selected_umap_test_df = selected_umap_df[selected_umap_df['split_label'].isin(['test_normal', 'test_anomaly'])].copy()
    if selected_umap_test_df.empty:
        print('[WARNING] The saved UMAP CSV does not contain test_normal/test_anomaly rows.')
    else:
        coordinates = selected_umap_test_df[['umap_1', 'umap_2']].to_numpy()
        labels = selected_umap_test_df['is_anomaly'].to_numpy()
        neighbor_count = min(16, len(selected_umap_test_df))
        if neighbor_count <= 1:
            print('[WARNING] Not enough UMAP points are available for neighborhood analysis.')
        else:
            nbrs = NearestNeighbors(n_neighbors=neighbor_count, metric='euclidean')
            nbrs.fit(coordinates)
            _, indices = nbrs.kneighbors(coordinates)
            neighbor_idx = indices[:, 1:]
            neighbor_labels = labels[neighbor_idx]
            selected_umap_test_df['anomaly_neighbor_ratio'] = neighbor_labels.mean(axis=1)
            selected_umap_test_df['normal_neighbor_ratio'] = 1.0 - selected_umap_test_df['anomaly_neighbor_ratio']
            embedded_anomaly_threshold = 0.8
            embedded_anomalies = selected_umap_test_df[(selected_umap_test_df['is_anomaly'] == 1) & (selected_umap_test_df['normal_neighbor_ratio'] >= embedded_anomaly_threshold)].copy()
            print(f"Total test anomalies: {(selected_umap_test_df['is_anomaly'] == 1).sum()}")
            print(f'Embedded anomalies (>= {embedded_anomaly_threshold:.0%} normal neighbors): {len(embedded_anomalies)}')
            display(selected_umap_test_df.head())


[WARNING] UMAP neighborhood analysis is unavailable because no saved UMAP CSV was found.


In [15]:
if selected_umap_test_df is None or selected_umap_test_df.empty:
    print('[WARNING] No UMAP scatter plot was generated because the saved UMAP test view is unavailable.')
else:
    fig, ax = plt.subplots(figsize=(9, 7))
    normals = selected_umap_test_df[selected_umap_test_df['is_anomaly'] == 0]
    anomalies = selected_umap_test_df[selected_umap_test_df['is_anomaly'] == 1]
    ax.scatter(normals['umap_1'], normals['umap_2'], s=10, alpha=0.18, label='test_normal')
    ax.scatter(anomalies['umap_1'], anomalies['umap_2'], s=14, alpha=0.45, label='test_anomaly')
    if not embedded_anomalies.empty:
        ax.scatter(embedded_anomalies['umap_1'], embedded_anomalies['umap_2'], s=28, alpha=0.95, marker='x', label='embedded_anomaly')
    ax.set_title('UMAP with Embedded Anomalies Highlighted')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


[WARNING] No UMAP scatter plot was generated because the saved UMAP test view is unavailable.
